In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time
import os
import pandas as pd 
from io import StringIO
import numpy as np
from bs4 import BeautifulSoup

In [ ]:
# Path to chromedriver executable
chromedriver_path = "chromedriver.exe"

# Initialize ChromeDriver with Selenium
service = Service(chromedriver_path)
driver = webdriver.Chrome(service=service)

In [ ]:
data_dir = "data"
standings_dir = os.path.join(data_dir, "standings")
scores_dir = os.path.join(data_dir, "scores")

In [ ]:
def get_html(url, selector, sleep_time=5):
    driver.get(url)
    time.sleep(sleep_time)
    element = driver.find_element(By.CSS_SELECTOR, selector)

    html_content = element.get_attribute("outerHTML")

    return html_content

In [ ]:
# List of seasons
seasons = [2025]

# Iterate over seasons
for season in seasons:
    # Construct URL
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_games.html"
    
    # Get the HTML content
    driver.get(url)
    
    # Wait for the page to load completely (adjust wait time as needed)
    time.sleep(10)
    
    # Find the element with the specified selector
    element = driver.find_element(By.CSS_SELECTOR, "#content .filter")
    
    # Extract HTML content of the element
    html_content = element.get_attribute("outerHTML")
    
    # Print HTML content
    print(f"HTML content for season {season} with selector '#content .filter':")
    print(html_content)
    print("="*50)

# Quit the driver
# driver.quit()

In [ ]:
soup = BeautifulSoup(html_content)
links = soup.find_all("a")

In [ ]:
links

In [ ]:
standings_pages = [f"https://www.basketball-reference.com{l['href']}" for l in links]

In [ ]:
standings_pages

In [ ]:
for url in standings_pages:
    save_path = os.path.join(standings_dir, url.split("/")[-1])
    if os.path.exists(save_path):
        continue

    with open(save_path, "w+") as f:
        f.write(html_content)

In [ ]:
save_path

In [ ]:
for url in standings_pages:
    save_path = os.path.join(standings_dir, url.split("/")[-1])
    if os.path.exists(save_path):
    
        driver.get(url)
        
        time.sleep(10)
        
        element = driver.find_element(By.CSS_SELECTOR, "#all_schedule")
        
        html_content = element.get_attribute("outerHTML")
        
        with open(save_path, "w+") as f:
            f.write(html_content)

In [ ]:
standings_files = os.listdir(standings_dir)

In [ ]:
filepath = os.path.join(standings_dir, standings_files[0])
with open(filepath, "r") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

In [ ]:
games = []

for game in soup.find("tbody").find_all("tr"):
    celda = game.find_all(["th", "td"])
    # print(celda[0].text)
    try:
        for i, c in enumerate(celda):
            # print(celda)
            games.append({
                "start_date": celda[0].get_text(strip=True),
                "start_time": celda[1].get_text(strip=True),
                "visitor_team": celda[2].get_text(strip=True),
                "visitor_pts": celda[3].get_text(strip=True),
                "home_team": celda[4].get_text(strip=True),
                "home_pts": celda[5].get_text(strip=True),
                # "link_box_score": celda[6].find("a")["href"] if c[6].find("a") else None,
                "attendace": celda[8].get_text(strip=True),
                "game_time": celda[9].get_text(strip=True),
                "stadium": celda[10].get_text(strip=True)
            })
    except Exception as e:
        print(f"Error processing game: {e}")


games_df = pd.DataFrame(games)

In [ ]:
games_df

In [ ]:
for standings_file in standings_files:
    filepath = os.path.join(standings_dir, standings_file)
    with open(filepath, 'r') as f:  
        html = f.read()

    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    hrefs = [l.get('href') for l in links]
    box_scores = [f"https://www.basketball-reference.com{l}" for l in hrefs if l and "boxscore" in l and '.html' in l]

    for url in box_scores:
        save_path = os.path.join(scores_dir, url.split("/")[-1])
        if os.path.exists(save_path):
            continue
        driver.get(url)
        print('Fetching Data')
        
        time.sleep(30)
        
        element = driver.find_element(By.CSS_SELECTOR, "#content")
        
        html_content = element.get_attribute("outerHTML")
        
        if not html_content:
            continue
        with open(save_path, "w+", encoding="utf-8") as f:
            f.write(html_content)
        

In [ ]:
box_scores = os.listdir(scores_dir)
box_scores = [os.path.join(scores_dir, f) for f in box_scores if f.endswith(".html")]

In [ ]:
directory = 'data/scores'

# List all files in the directory
files = os.listdir(directory)

# Filter HTML files
html_files = [file for file in files if file.endswith('.html')]

# Check if each HTML file is empty
empty_html_files = []
for file in html_files:
    file_path = os.path.join(directory, file)
    if os.path.getsize(file_path) == 0:
        empty_html_files.append(file)

# Print empty HTML files
print("Empty HTML files:")
for file in empty_html_files:
    print(file)

In [ ]:
len(box_scores)

In [ ]:
def parse_html(box_score):
    with open(box_score, encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html)
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thead")]
    return soup

In [ ]:
def read_season_info(soup):
    nav = soup.select("#bottom_nav_container")[0]
    hrefs = [a["href"] for a in nav.find_all('a')]
    season = os.path.basename(hrefs[1]).split("_")[0]
    return season

In [ ]:
def read_line_score(soup):
    line_score = pd.read_html(StringIO(str(soup)), attrs = {'id': 'line_score'})[0]

    score_divs = soup.find_all('div', string=True)
    scores = [div.text.strip() for div in score_divs if '-' in div.text]
    win_loss = [(int(score.split('-')[0]), int(score.split('-')[1])) for score in scores]
    df = pd.DataFrame(win_loss, columns=['W', 'L'])
    
    cols = list(line_score.columns)
    cols[0] = "team"
    cols[-1] = "total"
    cols[1] = 'Q1'
    cols[2] = 'Q2'
    cols[3] = 'Q3'
    cols[4] = 'Q4'
    line_score.columns = cols

    line_score = line_score[["team", "total", "Q1", "Q2", "Q3", "Q4"]]

    
    return pd.concat([line_score, df], axis=1)

In [ ]:
def read_stats(soup, team, stat):
    df = pd.read_html(StringIO(str(soup)), attrs = {'id': f'box-{team}-game-{stat}'}, index_col=0)[0]
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

In [ ]:
def read_player_stats(soup, team, stat):
    df = pd.read_html(StringIO(str(soup)), attrs = {'id': f'box-{team}-game-{stat}'}, index_col=0)[0]
    return df

In [ ]:
games = []
base_cols = None
i = 0
for box_score in box_scores:
    try:
        soup = parse_html(box_score)

        line_score = read_line_score(soup)
        teams = list(line_score["team"])

        summaries = []
        for team in teams:
            basic = read_stats(soup, team, "basic")
            advanced = read_stats(soup, team, "advanced")

            totals = pd.concat([basic.iloc[-1,:], advanced.iloc[-1,:]])
            totals.index = totals.index.str.lower()

            maxes = pd.concat([basic.iloc[:-1].max(), advanced.iloc[:-1].max()])
            maxes.index = maxes.index.str.lower() + "_max"

            summary = pd.concat([totals, maxes])
            
            if base_cols is None:
                base_cols = list(summary.index.drop_duplicates(keep="first"))
                base_cols = [b for b in base_cols if "bpm" not in b]
            
            summary = summary[base_cols]
            
            summaries.append(summary)
        summary = pd.concat(summaries, axis=1).T

        game = pd.concat([summary, line_score], axis=1)

        game["home"] = [0,1]

        game_opp = game.iloc[::-1].reset_index()
        game_opp.columns += "_opp"

        full_game = pd.concat([game, game_opp], axis=1)
        full_game["season"] = read_season_info(soup)
        
        full_game["date"] = os.path.basename(box_score)[:8]
        full_game["date"] = pd.to_datetime(full_game["date"], format="%Y%m%d")
        
        full_game["won"] = full_game["total"] > full_game["total_opp"]
        full_game['match_id'] = i
        i += 1
        games.append(full_game)
        
        if len(games) % 100 == 0:
            print(f"{len(games)} / {len(box_scores)}")
    except:
        continue

In [ ]:
games_df = pd.concat(games, ignore_index=True)

In [ ]:
games_df

In [ ]:
games_df['won'] = games_df['won'].astype(int)

In [ ]:
games_df.to_csv("nba_games.csv")

In [ ]:
player_games = []
base_cols = None
i = 0
for box_score in box_scores:
    try:
        home = 0
        soup = parse_html(box_score)

        line_score = read_line_score(soup)
        teams = list(line_score["team"])

        summaries = []
        for team in teams:
            opp = [t for t in teams if t != team][0]
            basic = read_player_stats(soup, team, "basic")
            advanced = read_player_stats(soup, team, "advanced")

            summary = pd.concat([basic.iloc[:-1, :], advanced.iloc[:-1, 1:]], axis=1)
            summary['team'] = team
            summary['home'] = home
            summary['opp_team'] = opp
            summary['total_team_points'] = line_score.loc[line_score['team']==team]['total'].values[0]
            summary['total_opp_points'] = line_score.loc[line_score['team']==opp]['total'].values[0]
            summary['won'] = summary['total_team_points'] > summary['total_opp_points']

            home += 1           
            summaries.append(summary)

        players = pd.concat(summaries)
        players.reset_index(inplace=True)
        players.rename(columns={'Starters': 'players'}, inplace=True)

        players['match_id'] = i
        
        players["date"] = os.path.basename(box_score)[:8]
        players["date"] = pd.to_datetime(players["date"], format="%Y%m%d")
        
        player_games.append(players)

        i += 1
        
        if len(games) % 100 == 0:
            print(f"{len(games)} / {len(box_scores)}")
    except:
        continue

In [ ]:
player_games

In [ ]:
players_df = pd.concat(player_games)
players_df.reset_index(inplace=True)
players_df.drop(columns=['index'], inplace=True)
players_df

In [ ]:
players_df.replace('Did Not Play', np.nan, inplace=True)
players_df.replace('Not With Team', np.nan, inplace=True)
players_df.replace('Player Suspended', np.nan, inplace=True)
players_df.replace('Did Not Dress', np.nan, inplace=True)

In [ ]:
players_df['MP'].fillna('00:00', inplace=True)

In [ ]:
players_df[['Minutes', 'Seconds']] = players_df['MP'].str.split(':', expand=True)

# Convert minutes and seconds to seconds and sum them up
players_df['MP'] = pd.to_numeric(players_df['Minutes']) * 60 + pd.to_numeric(players_df['Seconds'])

# Drop the 'Minutes' and 'Seconds' columns if needed
players_df.drop(columns=['Minutes', 'Seconds'], inplace=True)
players_df.rename(columns={'MP': 'time(s)'}, inplace=True)

In [ ]:
players_df['won'] = players_df['won'].astype(int)

In [ ]:
new_types = {"FG" : float,
"FGA" : float,
"FG%" : float,
"3P" : float,
"3PA" : float,
"3P%" : float,
"FT" : float,
"FTA" : float,
"FT%" : float,
"ORB" : float,
"DRB" : float,
"TRB" : float,
"AST" : float,
"STL" : float,
"BLK" : float,
"TOV" : float,
"PF" : float,
"PTS" : float,
"+/-" : float,
"TS%" : float,
"eFG%" : float,
"3PAr" : float,
"FTr" : float,
"ORB%" : float,
"DRB%" : float,
"TRB%" : float,
"AST%" : float,
"STL%" : float,
"BLK%" : float,
"TOV%" : float,
"USG%" : float,
"ORtg" : float,
"DRtg" : float,
"BPM" : float,
"opp_team":str}

In [ ]:
new_column_order = [ 
 'match_id',
 'date', 'players',  'team',
 'time(s)',
 'FG',
 'FGA',
 'FG%',
 '3P',
 '3PA',
 '3P%',
 'FT',
 'FTA',
 'FT%',
 'ORB',
 'DRB',
 'TRB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PTS',
 '+/-',
 'TS%',
 'eFG%',
 '3PAr',
 'FTr',
 'ORB%',
 'DRB%',
 'TRB%',
 'AST%',
 'STL%',
 'BLK%',
 'TOV%',
 'USG%',
 'ORtg',
 'DRtg',
 'BPM',
 'home',
 'opp_team',
 'total_team_points',
 'total_opp_points','won'
]

In [ ]:
players_df = players_df[new_column_order]

In [ ]:
players_df = players_df.astype(new_types)

In [ ]:
counting_stats = ['FG', 'FGA', 'FG%', '3P', '3PA', '3P%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS' ]

In [ ]:
players_df.loc[players_df['players'] == 'Lamar Stevens'][counting_stats]

In [ ]:
players_df.loc[(players_df['team'] == 'BOS') & (players_df['opp_team'] == 'PHI')]

In [ ]:
players_df.to_csv("nba_games_players.csv")